# HR Analytics

Summarise workforce size, pay, performance, hiring and attrition using the employee dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('../data/hr_employee_data_18000.csv')
display(df.head())
print('Shape:', df.shape)

In [ ]:
print('Missing values by column:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Attrition values:\n', df['Attrition'].value_counts(dropna=False))
if df['Attrition'].isna().any() or not set(df['Attrition'].dropna().unique()).issubset({0, 1}):
    raise ValueError('Attrition must contain non-missing binary values (0/1).')
df = df.drop_duplicates().copy()
kpis = {
    'Employees': len(df),
    'Attrition Rate %': df['Attrition'].mean() * 100,
    'Average Salary': df['Salary'].mean(),
    'Average Performance': df['Performance_Rating'].mean()
}
display(pd.Series(kpis))

In [ ]:
department = df.groupby('Department').agg(
    Employees=('Employee_ID', 'nunique'),
    Attrition_Rate=('Attrition', 'mean'),
    Average_Salary=('Salary', 'mean'),
    Average_Performance=('Performance_Rating', 'mean')
)
department['Attrition_Rate'] *= 100
overtime = df.groupby('Overtime').agg(
    Employees=('Employee_ID', 'nunique'), Attrition_Rate=('Attrition', 'mean')
)
overtime['Attrition_Rate'] *= 100
tenure = df.groupby('Tenure_Years').agg(
    Employees=('Employee_ID', 'nunique'), Attrition_Rate=('Attrition', 'mean')
)
tenure['Attrition_Rate'] *= 100
salary_band = pd.qcut(df['Salary'], q=4, duplicates='drop')
salary_attrition = df.groupby(salary_band, observed=False)['Attrition'].mean().mul(100)
hiring = df.groupby('Hire_Year').size().rename('Hires')
display(department.sort_values('Attrition_Rate', ascending=False))
display(overtime)
display(tenure)
display(salary_attrition)
display(hiring)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
hiring.plot(kind='bar', ax=ax)
ax.set(title='Employees hired by year', xlabel='Hire year', ylabel='Employees')
fig.tight_layout()
plt.show()

## Interpretation

Use department, tenure, overtime and salary-band comparisons to decide where further investigation may be useful. These are descriptive comparisons. They do not establish that salary, overtime or tenure caused an employee to leave. Check group sizes before drawing conclusions from small segments.

## Power BI handoff

Suggested KPIs: employee count, attrition rate, average salary and average performance rating. Useful views include attrition by department, tenure, overtime and salary band, plus hiring trends by year.